# EGO Step2 F0 — Colab A100 GPU 스모크

F0 (WM-only GRPO) 학습 코드가 이 GPU 에서 **처음부터 끝까지 도는지**를 검증한다.
학습 품질/수렴은 검증 대상이 아니다 (그건 실데이터 + full run 의 몫).

런타임: **A100 40GB** 권장 (런타임 → 런타임 유형 변경 → GPU: A100).

두 모드:
- `--wm synth` (기본): 다운로드 없이 스키마-정합 합성 jsonl 로 Qwen3-VL GRPO 루프만 검증. 수 분.
- `--wm real`: 공식 V-JEPA2 ViT-g/384 백본 + EK100 AC probe 를 실제로 받아 GPU 로드·forward → 실 후보 산출 → GRPO 루프. 다운로드 ~수 GB (백본이 큼).

In [ ]:
# 1) GPU 확인
!nvidia-smi -L

In [ ]:
# 2) 리포 준비 (push 된 브랜치를 clone). 새 러너 파일이 커밋돼 있어야 한다.
%cd /content
!git clone https://github.com/hublemon/EGO.git 2>/dev/null || (cd EGO && git pull)
%cd /content/EGO

## 모드 A — 합성 데이터 스모크 (기본, 다운로드 없음)

모델 로드 → 생성 → WM-likelihood reward → LoRA 업데이트 → 체크포인트까지 검증.

In [ ]:
!python scripts/step2/colab_smoke_f0.py --wm synth

## 모드 B — 공식 V-JEPA2 실추론 스모크 (선택)

백본/probe 를 매 세션 다시 받지 않으려면 `--assets_dir` 를 Google Drive 로 지정한다.
아래 mount 셀은 선택 — Drive 캐시를 쓸 때만 실행.

In [ ]:
# (선택) Drive 마운트 — 체크포인트 캐시 재사용
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!python scripts/step2/colab_smoke_f0.py --wm real \
    --assets_dir /content/drive/MyDrive/ego_vjepa_assets

## 모드 C — 실데이터 validation/full 학습 (smoke 통과 후)

전제: v2 데이터 빌드 산출물(`grpo_train.jsonl` + 4f grid JPEG)이 Drive 에 업로드돼 있어야 한다.
jsonl 의 `image_path` 가 서버 절대경로면 `--path_map "서버프리픽스=콜랩프리픽스"` 로 재작성.

- A100 40GB 등가 변환: per_device 2 × accum 4 = 스텝당 프롬프트 1개 × gen 8 (gen 8 은 동결값)
- `--out_dir` 는 Drive 로 — 세션이 끊겨도 체크포인트(125 스텝마다)가 남는다
- 끊긴 뒤 재개: `--adapter_path <out_dir>/checkpoint-<N>` (가중치만 이어받음, 옵티마이저 초기화)
- full(1 epoch ≈ 5000 스텝)은 단일 A100 에서 세션 한도 초과 가능성 큼 → Colab 은 validation 권장

In [ ]:
# validation run (500 스텝, 레시피 동결값 유지). 먼저 Drive 마운트 셀 실행 필요.
!python scripts/step2/colab_train_f0_full.py \
    --train_jsonl /content/drive/MyDrive/ego/grpo_train.jsonl \
    --path_map "/path/on/server/frames=/content/drive/MyDrive/ego/frames" \
    --out_dir /content/drive/MyDrive/ego/runs/f0_v2_val_a100 \
    --run_mode validation --copy_frames_local

In [ ]:
# (세션 끊긴 뒤 재개 예시 — 마지막 체크포인트에서 가중치만 이어받아 잔여 스텝 학습)
# !python scripts/step2/colab_train_f0_full.py \
#     --train_jsonl /content/drive/MyDrive/ego/grpo_train.jsonl \
#     --path_map "/path/on/server/frames=/content/drive/MyDrive/ego/frames" \
#     --out_dir /content/drive/MyDrive/ego/runs/f0_v2_val_a100_resume \
#     --adapter_path /content/drive/MyDrive/ego/runs/f0_v2_val_a100/checkpoint-375 \
#     --run_mode validation --max_steps 125 --copy_frames_local --no_install

## 모드 D — 10시간 무인 파이프라인 (VSCode Colab 커널 권장)

**VSCode 에서 실행하는 법**: 이 노트북을 VSCode 로 열고 → 우상단 커널 선택 → Colab → A100 런타임.
셀은 원격 Colab VM(/content)에서 실행된다.

- **Drive 마운트 주의**: `drive.mount` 의 인증 팝업은 VSCode 에서 안 뜰 수 있다.
  해결: 같은 런타임을 **브라우저 Colab 에서 한 번 열어** 마운트하면(런타임 공유) VSCode 에서도 /content/drive 가 보인다.
- **10시간 안정성**: 파이프라인은 `nohup ... &` 백그라운드로 띄운다 — VSCode/커널 클라이언트가 끊겨도
  프로세스는 계속 돈다. 런타임 자체가 회수될 때를 대비해 `--out_root` 는 반드시 Drive.
- 파이프라인 구성: 4f-base 게이트 평가 → 500-step 검증 run(데드라인 자동 중단·125step 체크포인트)
  → trained 평가 → (잔여 시간) no_memory 평가 → session_report.md

In [ ]:
# 10h 무인 파이프라인 — 백그라운드 실행 (한 번만 실행)
!cd /content/EGO && nohup python scripts/step2/colab_f0_10h_pipeline.py \
    --train_jsonl  /content/drive/MyDrive/ego/grpo_train.jsonl \
    --heldout_jsonl /content/drive/MyDrive/ego/grpo_heldout.jsonl \
    --path_map "/path/on/server/frames=/content/drive/MyDrive/ego/frames" \
    --out_root /content/drive/MyDrive/ego/runs/f0_v2_session1 \
    --budget_hours 10 --copy_frames_local > /content/pipeline.log 2>&1 &
print("started — 아래 모니터링 셀로 확인")

In [ ]:
# 모니터링 — 필요할 때마다 실행 (VSCode 가 끊겼다 재접속해도 동작)
!tail -n 30 /content/pipeline.log
!ls /content/drive/MyDrive/ego/runs/f0_v2_session1/train_val500/ 2>/dev/null | grep checkpoint
!tail -n 3 /content/drive/MyDrive/ego/runs/f0_v2_session1/train_val500/reward_log.jsonl 2>/dev/null

## 참고

- OOM 이면 더 작은 정책모델로: `--model Qwen/Qwen2.5-VL-7B-Instruct`
- 실데이터 jsonl 이 있으면: `--train_jsonl /content/drive/MyDrive/.../grpo_train.jsonl`
- 규모 조절 플래그: `--n_samples --max_steps --num_generations --per_device_batch --max_completion_length`
- 이미 설치된 환경이면: `--no_install`